# Prepare Data on Server
Chuẩn bị dữ liệu trên server: Tải từ S3, xử lý mây, tính NDVI, lưu thành file

**Sau khi chạy xong, tải các file data xuống máy cá nhân để train model**

In [ ]:
%%time
%matplotlib inline

import importlib
import new_import_ODC  

importlib.reload(new_import_ODC)

from new_import_ODC import *

In [ ]:
%%time
# Cấu hình Daskgateway
cluster, client = notebook_utils.initialize_dask(use_gateway=True, workers=(1, 10))
# Khai báo 1 Datacube là dc
dc = datacube.Datacube()

# Cấu hình truy cập dịch vụ S3
configure_s3_access(aws_unsigned=False, requester_pays=True, client=client)

client

In [ ]:
## Cấu hình thời gian lấy ảnh và tọa độ
date_range = ("2022-09-01", "2023-10-01")
longtitude_range = (105.5, 106.4)
latitude_range = (9.2, 10.0)

coordinates = (longtitude_range, latitude_range)

In [ ]:
## DEBUG: Inspect what datacube wants to load
print("🔍 DIAGNOSTIC: Checking datacube metadata...\n")

# Check available products
available_products = dc.list_products()
s2_products = available_products[available_products['name'].str.contains('s2', case=False)]
print(f"Available S2 products:\n{s2_products[['name', 'description']].to_string()}\n")

# Query to check what would be loaded
test_query = {
    'product': 's2_l2a',
    'x': longtitude_range,
    'y': latitude_range,
    'time': ("2023-01-01", "2023-02-01"),  # Just 1 month for testing
}

print(f"Test query: {test_query}")

try:
    # This queries metadata only, doesn't load data
    test_datasets = dc.find_datasets(**test_query)
    print(f"\n📊 Metadata check for Jan 2023:")
    print(f"   Found {len(test_datasets)} scenes")
    if test_datasets:
        first_ds = test_datasets[0]
        print(f"   First scene: {first_ds.center_time}")
        print(f"   Bounds: {first_ds.bounds}")
        print(f"   CRS: {first_ds.crs}")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "="*60 + "\n")

In [ ]:
## SENTINEL-2 LOADING: Monthly chunks to prevent OOM
print("📡 Tải dữ liệu Sentinel-2 L2A từ S3...")
print(f"  AOI: {longtitude_range}, {latitude_range}")
print(f"  Time range: {date_range}\n")

data = None

# Strategy: Load 13 monthly chunks instead of 396 scenes at once
# This keeps memory usage manageable (~5-15 GB per month)

date_ranges = [
    ("2022-09-01", "2022-10-01"),
    ("2022-10-01", "2022-11-01"),
    ("2022-11-01", "2022-12-01"),
    ("2022-12-01", "2023-01-01"),
    ("2023-01-01", "2023-02-01"),
    ("2023-02-01", "2023-03-01"),
    ("2023-03-01", "2023-04-01"),
    ("2023-04-01", "2023-05-01"),
    ("2023-05-01", "2023-06-01"),
    ("2023-06-01", "2023-07-01"),
    ("2023-07-01", "2023-08-01"),
    ("2023-08-01", "2023-09-01"),
    ("2023-09-01", "2023-10-01"),
]

product = 's2_l2a'
measurements = ['red', 'nir', 'scl']

# Get native CRS once
try:
    query_crs = {
        'product': product,
        'x': longtitude_range,
        'y': latitude_range,
        'time': date_range,
    }
    native_crs = notebook_utils.mostcommon_crs(dc, query_crs)
    print(f"✅ Native CRS: {native_crs}\n")
except Exception as e:
    print(f"⚠️  Could not determine CRS: {e}")
    native_crs = 'EPSG:32648'  # Fallback for UTM Zone 48N

data_list = []

for i, (start_date, end_date) in enumerate(date_ranges):
    print(f"[{i+1:2d}/13] {start_date} → {end_date}  ", end="", flush=True)
    
    try:
        monthly_query = {
            'product': product,
            'x': longtitude_range,
            'y': latitude_range,
            'time': (start_date, end_date),
        }
        
        load_params = {
            'measurements': measurements,
            'output_crs': native_crs,
            'resolution': (-10, 10),
            'group_by': 'solar_day',
            'dask_chunks': {'x': 512, 'y': 512, 'time': 1},
            'skip_broken_datasets': True,
        }
        
        monthly_data = load_s2l2a_with_offset(dc, monthly_query | load_params)
        
        n_scenes = monthly_data.sizes['time']
        if n_scenes > 0:
            data_list.append(monthly_data)
            print(f"✓ {n_scenes} scenes")
        else:
            print("⚠️  0 scenes")
    
    except MemoryError as e:
        print(f"❌ OOM: {str(e)[:60]}")
        break
    except Exception as e:
        print(f"❌ {str(e)[:60]}")
        continue

# Combine all monthly chunks
if data_list:
    print(f"\n🔗 Combining {len(data_list)} monthly chunks...")
    data = xr.concat(data_list, dim='time')
    print(f"✅ Success! Shape: {dict(data.dims)}")
    print(f"   Memory: {notebook_utils.xarray_object_size(data)}")
    display(data)
else:
    print("\n❌ Failed to load any scenes")

In [ ]:
%%time
# Loại bỏ các vị trí bị mây ảnh hưởng
print("☁️  Xử lý mây...")
result = mask_clean(data)
progress(result)

In [ ]:
# Tính toán NDVI
print("🌱 Tính NDVI...")
ds1 = calculate_indices(result, index="NDVI", satellite_mission="s2")
ndvi = ds1["NDVI"]
print(f"✅ NDVI shape: {ndvi.shape}")

In [ ]:
# Điền mây sử dụng seasonal interpolation
print("🔧 Điền mây theo mùa vụ...")
time_split = [
    slice("2022-09-01", "2023-01-01"),
    slice("2023-01-01", "2023-05-01"),
    slice("2023-05-01", "2023-07-01"),
    slice("2023-07-01", "2023-10-01"),
]

fill_nan_ndvi = fill_nan(ndvi, time_split)
print(f"✅ Mây đã được điền")

In [ ]:
%%time
# Tính NDVI theo tháng
print("📊 Tính NDVI trung bình theo tháng...")
average_ndvi = fill_nan_ndvi.resample(time="1M").mean().persist()
progress(average_ndvi)
average_ndvi = average_ndvi.compute()
print(f"✅ NDVI theo tháng shape: {average_ndvi.shape}")

In [ ]:
# Tải dữ liệu Sentinel-1 (VH, VV)
print("📡 Tải dữ liệu Sentinel-1 từ S3...")
dsvh, dsvv = load_data_sen1(dc, date_range, coordinates)
print("📊 Tính VV, VH trung bình theo tháng...")
average_vv = calculate_average(dsvv, time_pattern='1M')
average_vh = calculate_average(dsvh, time_pattern='1M')
print(f"✅ VV shape: {average_vv.shape}")
print(f"✅ VH shape: {average_vh.shape}")

## Lưu dữ liệu đã xử lý thành file NetCDF

In [ ]:
import os

# Tạo thư mục lưu data
data_dir = "data_for_training"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f"✅ Tạo thư mục {data_dir}")

# Lưu NDVI
ndvi_path = os.path.join(data_dir, "average_ndvi.nc")
print(f"💾 Lưu NDVI vào {ndvi_path}...")
average_ndvi.to_netcdf(ndvi_path)
print(f"✅ NDVI đã lưu ({os.path.getsize(ndvi_path) / 1024**2:.2f} MB)")

# Lưu VV
vv_path = os.path.join(data_dir, "average_vv.nc")
print(f"💾 Lưu VV vào {vv_path}...")
average_vv.to_netcdf(vv_path)
print(f"✅ VV đã lưu ({os.path.getsize(vv_path) / 1024**2:.2f} MB)")

# Lưu VH
vh_path = os.path.join(data_dir, "average_vh.nc")
print(f"💾 Lưu VH vào {vh_path}...")
average_vh.to_netcdf(vh_path)
print(f"✅ VH đã lưu ({os.path.getsize(vh_path) / 1024**2:.2f} MB)")

print(f"\n✅ Tất cả dữ liệu đã lưu trong thư mục '{data_dir}'")
print(f"📥 Hãy tải các file này xuống máy cá nhân để train model")

In [ ]:
# Lưu training data
train_path = "train/ST_training data_updated_1130points_new.shp"
import shutil

print(f"📋 Copy training data...")
# Copy toàn bộ các file liên quan đến shapefile
train_dir = "train"
train_output = os.path.join(data_dir, "train_data")
if not os.path.exists(train_output):
    os.makedirs(train_output)

for file in os.listdir(train_dir):
    if "1130points_new" in file:
        src = os.path.join(train_dir, file)
        dst = os.path.join(train_output, file)
        shutil.copy2(src, dst)
        print(f"✅ {file}")

print(f"\n✅ Training data đã copy vào '{train_output}'")

In [ ]:
# Đóng client, cluster
print("\n🔌 Đóng kết nối...")
client.close()
cluster.close()
print("✅ Xong!")